In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
# DECODING CUSTOMER VALUE — Step 1: EDA, Cleaning & Feature Engineering

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

DATA_PATH = "/kaggle/input/datasets/grachna/decoding-customer-value-strategy-project/Dataset.csv"  
df = pd.read_csv(DATA_PATH)

# PART 1: EXPLORATORY DATA ANALYSIS
# Goal: understand structure, find anomalies, and surface the structural

print("="*80)
print("1.1 BASIC STRUCTURE")
print("="*80)
print("Shape:", df.shape)
print("Customer ID unique:", df['Customer ID'].is_unique)
print("\nDtypes:\n", df.dtypes)
print("\nNulls:\n", df.isnull().sum())
print("\nFull duplicate rows:", df.duplicated().sum())

print("\n" + "="*80)
print("1.2 NUMERIC SUMMARY")
print("="*80)
print(df[['Age', 'Purchase Amount (USD)', 'Review Rating', 'Previous Purchases']].describe())

print("\n" + "="*80)
print("1.3 CATEGORICAL DISTRIBUTIONS")
print("="*80)
cat_cols = ['Gender', 'Category', 'Season', 'Subscription Status', 'Shipping Type',
            'Discount Applied', 'Promo Code Used', 'Payment Method', 'Frequency of Purchases']
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())

print("\n" + "="*80)
print("1.4 STRUCTURAL ARTIFACT CHECKS (critical — these drive cleaning decisions)")
print("="*80)

# Check 1: Are Discount Applied and Promo Code Used actually two different fields?
match_rate = (df['Discount Applied'] == df['Promo Code Used']).mean()
print(f"Discount Applied == Promo Code Used match rate: {match_rate:.2%}")
print(" -> If ~100%, these are duplicate fields. Keep one, drop the other, document why.")

# Check 2: Is Discount Applied actually a behavioral signal, or a structural flag?
print("\nDiscount Applied rate by Gender:")
print(df.groupby('Gender')['Discount Applied'].apply(lambda x: (x == 'Yes').mean()))

print("\nDiscount Applied rate by Subscription Status:")
print(df.groupby('Subscription Status')['Discount Applied'].apply(lambda x: (x == 'Yes').mean()))

print("\nSubscription Status by Gender (check if subscription itself is gender-confounded):")
print(pd.crosstab(df['Gender'], df['Subscription Status'], normalize='index'))
print(" -> If discount usage is ~0%/100% split by gender or subscription, it is NOT a")
print("    customer choice signal in this dataset — it's a generation artifact.")
print("    Decision: exclude raw Discount Applied from value/loyalty scoring, or only")
print("    use it stratified within gender. State this explicitly as a data limitation.")

# Check 3: Are Bi-Weekly and Fortnightly being treated as different cadences
# when they mean the same thing (every two weeks)?
print("\nFrequency of Purchases categories (check for duplicate-meaning labels):")
print(df['Frequency of Purchases'].value_counts())
print(" -> 'Bi-Weekly' and 'Fortnightly' both mean every 2 weeks. Must be merged")
print("    before converting frequency to a numeric scale, or you'll artificially")
print("    split your highest-cadence segment in two.")

print("\n" + "="*80)
print("1.5 CORRELATION CHECK (sets expectations for feature engineering)")
print("="*80)
print(df[['Age', 'Purchase Amount (USD)', 'Review Rating', 'Previous Purchases']].corr())
print(" -> If these are all near-zero, no single raw variable will tell the loyalty")
print("    story on its own. Composite/engineered features are necessary, not optional.")

print("\n" + "="*80)
print("1.6 MAIN-EFFECT CHECKS: Category / Season / Geography vs tenure & spend")
print("="*80)
print("\nAvg Previous Purchases by Category:")
print(df.groupby('Category')['Previous Purchases'].mean().sort_values(ascending=False))
print("\nAvg Previous Purchases by Season:")
print(df.groupby('Season')['Previous Purchases'].mean().sort_values(ascending=False))
print("\nAvg spend by Location (top 8 / bottom 8) — flag if sample sizes are small:")
loc_summary = df.groupby('Location').agg(
    n=('Customer ID', 'count'),
    avg_spend=('Purchase Amount (USD)', 'mean'),
    avg_prev_purchases=('Previous Purchases', 'mean')
).sort_values('avg_spend', ascending=False)
print(loc_summary.head(8))
print(loc_summary.tail(8))
print(" -> ~50 states with ~60-90 customers each means state-level claims are noisy.")
print("    Roll up to region for the 'geographic opportunity' analysis; use state only")
print("    for illustrative drill-down, with sample size caveated.")

print("\n" + "="*80)
print("1.7 MISSINGNESS PATTERN — Review Rating")
print("="*80)
print("Missing count:", df['Review Rating'].isnull().sum())
print("Missingness by Category (check if missing is random or structural):")
print(df[df['Review Rating'].isnull()]['Category'].value_counts(normalize=True))
print(df['Category'].value_counts(normalize=True))
print(" -> Compare the two distributions above. If similar, missingness looks random")
print("    -> safe to impute. If skewed, document as a limitation.")



1.1 BASIC STRUCTURE
Shape: (3900, 18)
Customer ID unique: True

Dtypes:
 Customer ID                 int64
Age                         int64
Gender                     object
Item Purchased             object
Category                   object
Purchase Amount (USD)       int64
Location                   object
Size                       object
Color                      object
Season                     object
Review Rating             float64
Subscription Status        object
Shipping Type              object
Discount Applied           object
Promo Code Used            object
Previous Purchases          int64
Payment Method             object
Frequency of Purchases     object
dtype: object

Nulls:
 Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Re

In [5]:
# PART 2: CLEANING

df_clean = df.copy()

# Decision 1: Discount Applied and Promo Code Used are identical -> drop one.
# Keep 'Discount Applied' as canonical; drop 'Promo Code Used' to avoid double-
# counting the same signal in any downstream score.
assert (df_clean['Discount Applied'] == df_clean['Promo Code Used']).mean() > 0.99, \
    "Re-check: these were expected to be near-identical."
df_clean = df_clean.drop(columns=['Promo Code Used'])

# Decision 2: Merge Bi-Weekly and Fortnightly (same real-world cadence) before
# building any numeric frequency feature.
df_clean['Frequency of Purchases'] = df_clean['Frequency of Purchases'].replace(
    {'Fortnightly': 'Bi-Weekly'}
)

# Decision 3: Review Rating — add a missingness flag (whether a customer bothered
# to rate at all can itself be a weak engagement signal), then impute with the
# category median so we don't lose 37 rows from analysis.
df_clean['Rating_Missing_Flag'] = df_clean['Review Rating'].isnull().astype(int)
df_clean['Review Rating'] = df_clean.groupby('Category')['Review Rating'] \
    .transform(lambda x: x.fillna(x.median()))

# Decision 4: Standardize Yes/No fields to boolean for cleaner downstream math.
for col in ['Discount Applied', 'Subscription Status']:
    df_clean[col + '_Flag'] = (df_clean[col] == 'Yes').astype(int)

print("\n" + "="*80)
print("2.1 CLEANING SUMMARY")
print("="*80)
print("Shape after cleaning:", df_clean.shape)
print("Remaining nulls:\n", df_clean.isnull().sum().sum())






2.1 CLEANING SUMMARY
Shape after cleaning: (3900, 20)
Remaining nulls:
 0


In [15]:
# PART 3: FEATURE ENGINEERING


# ---- 3.1 Estimated Lifetime Spend ----
# WHY: There's no true lifetime spend field — only a single observed order
# amount + a count of previous purchases. We approximate lifetime spend by
# assuming the observed order value is representative of the customer's
# typical order. This is a load-bearing assumption: state it explicitly in
# the report. It answers: "how much revenue has this customer likely
# generated for us so far?"
df_clean['Estimated_Lifetime_Spend'] = (
    df_clean['Purchase Amount (USD)'] * (df_clean['Previous Purchases'] + 1)
)

# ---- 3.2 Purchase Frequency (numeric, purchases/year) ----
# WHY: 'Frequency of Purchases' is an ordinal label, not a number, which makes
# it unusable in any score or ranking. Converting to an estimated annual
# purchase rate lets us combine it with spend and tenure. Answers: "how
# active is this customer, on a comparable numeric scale?"
freq_map = {
    'Weekly': 52,
    'Bi-Weekly': 26,
    'Monthly': 12,
    'Quarterly': 4,
    'Every 3 Months': 4,
    'Annually': 1
}
df_clean['Purchases_Per_Year'] = df_clean['Frequency of Purchases'].map(freq_map)

# ---- 3.3 Value Tier ----
# WHY: A continuous value score is hard for a non-technical founder to act on.
# Tiering (quartile-based) directly answers: "which customers should get
# white-glove treatment vs. which are low priority?" — a decision, not just
# a number. Built from Estimated Lifetime Spend, since that's the closest
# proxy we have to "how much has this person been worth to us."
df_clean['Value_Tier'] = pd.qcut(
    df_clean['Estimated_Lifetime_Spend'],
    q=4,
    labels=['Bronze', 'Silver', 'Gold', 'Platinum']
)

# ---- 3.4 Satisfaction Flag ----
# WHY: Answers a sharp, specific question the brand needs flagged: "do we
# have high-value customers who are quietly unhappy?" Rather than just
# reporting average rating, this creates an actionable binary the marketing
# team can filter on directly.
df_clean['Low_Satisfaction_Flag'] = (df_clean['Review Rating'] < 3.5).astype(int)

# ---- 3.5 Promo Dependency Score (handled carefully due to the gender/
# subscription artifact found in EDA) ----
# WHY: The brand explicitly wants to know who needs a discount to buy. But we
# found Discount Applied is structurally tied to Gender (0% for Female, ~63%
# for Male) and Subscription (100% when subscribed) — not a clean behavioral
# signal. Building this as a raw flag would let a data artifact masquerade as
# a business insight. Instead, we rank discount usage WITHIN gender, so the
# score reflects relative reliance among comparable customers, not the
# underlying structural skew. This must be stated as a limitation in the
# report regardless.
df_clean['Promo_Dependency_Score'] = df_clean.groupby('Gender')['Discount Applied_Flag'] \
    .rank(pct=True)

# ---- 3.6 Loyalty Definition A — Behavioral Loyalty ----
# WHY: Built purely from observed purchase behavior (tenure proxy x cadence),
# deliberately avoiding discount/subscription fields to sidestep the gender
# confound entirely. Answers: "based on how often and how long this customer
# has been buying, do they look like a habitual repeat customer?"
df_clean['Loyalty_Score_A_Behavioral'] = (
    df_clean['Previous Purchases'].rank(pct=True) * 0.5 +
    df_clean['Purchases_Per_Year'].rank(pct=True) * 0.5
)

# ---- 3.7 Loyalty Definition B — Value + Satisfaction Loyalty ----
# WHY: Tests a different lens — are big spenders also satisfied? Answers:
# "is this customer's value sustainable, or are they a high-spender who
# might churn because they're unhappy?" This surfaces a segment Definition A
# alone would miss: high spend, low satisfaction = flight risk.
df_clean['Loyalty_Score_B_Value_Satisfaction'] = (
    df_clean['Estimated_Lifetime_Spend'].rank(pct=True) * 0.5 +
    df_clean['Review Rating'].rank(pct=True) * 0.5
)

# ---- 3.8 Region rollup (for geography analysis) ----
# WHY: ~50 states with only 60-90 customers each is too noisy for state-level
# claims. Rolling up to census region gives a more statistically stable view
# for the "underlevered geography" question; state-level can still be used
# for illustrative drill-downs with sample-size caveats.
region_map = {
    'Connecticut': 'Northeast', 'Maine': 'Northeast', 'Massachusetts': 'Northeast',
    'New Hampshire': 'Northeast', 'Rhode Island': 'Northeast', 'Vermont': 'Northeast',
    'New Jersey': 'Northeast', 'New York': 'Northeast', 'Pennsylvania': 'Northeast',
    'Illinois': 'Midwest', 'Indiana': 'Midwest', 'Michigan': 'Midwest', 'Ohio': 'Midwest',
    'Wisconsin': 'Midwest', 'Iowa': 'Midwest', 'Kansas': 'Midwest', 'Minnesota': 'Midwest',
    'Missouri': 'Midwest', 'Nebraska': 'Midwest', 'North Dakota': 'Midwest', 'South Dakota': 'Midwest',
    'Delaware': 'South', 'Florida': 'South', 'Georgia': 'South', 'Maryland': 'South',
    'North Carolina': 'South', 'South Carolina': 'South', 'Virginia': 'South',
    'West Virginia': 'South', 'Alabama': 'South', 'Kentucky': 'South', 'Mississippi': 'South',
    'Tennessee': 'South', 'Arkansas': 'South', 'Louisiana': 'South', 'Oklahoma': 'South',
    'Texas': 'South',
    'Arizona': 'West', 'Colorado': 'West', 'Idaho': 'West', 'Montana': 'West',
    'Nevada': 'West', 'New Mexico': 'West', 'Utah': 'West', 'Wyoming': 'West',
    'Alaska': 'West', 'California': 'West', 'Hawaii': 'West', 'Oregon': 'West',
    'Washington': 'West'
}
df_clean['Region'] = df_clean['Location'].map(region_map)
print("\nUnmapped locations (check if any state names are missing from region_map):")
print(df_clean[df_clean['Region'].isnull()]['Location'].unique())

# Preview the newly engineered features at the customer level
preview_cols = [
    'Customer ID', 'Gender', 'Purchase Amount (USD)', 'Previous Purchases',
    'Frequency of Purchases', 'Purchases_Per_Year', 'Estimated_Lifetime_Spend',
    'Value_Tier', 'Review Rating', 'Low_Satisfaction_Flag',
    'Promo_Dependency_Score', 'Loyalty_Score_A_Behavioral',
    'Loyalty_Score_B_Value_Satisfaction', 'Region'
]
df_clean[preview_cols].head(15)

# Diagnostic only — NOT used to adjust Previous Purchases, since Frequency
# has no validated relationship to it (p=0.118 even controlling for
# Age/Gender/Category). This flags profiles worth a one-line callout in
# the report as a data quality limitation, e.g. "X% of customers report a
# cadence implausible given their purchase count."
implausible = (
    ((df_clean['Frequency of Purchases'] == 'Annually') & (df_clean['Previous Purchases'] > 20)) |
    ((df_clean['Frequency of Purchases'] == 'Weekly') & (df_clean['Previous Purchases'] < 5))
)
df_clean['Frequency_Inconsistency_Flag'] = implausible.astype(int)


Unmapped locations (check if any state names are missing from region_map):
[]


,Customer ID,Gender,Purchase Amount (USD),Previous Purchases,Frequency of Purchases,Purchases_Per_Year,Estimated_Lifetime_Spend,Value_Tier,Review Rating,Low_Satisfaction_Flag,Promo_Dependency_Score,Loyalty_Score_A_Behavioral,Loyalty_Score_B_Value_Satisfaction,Region
0,1,Male,53,14,Bi-Weekly,26,795,Silver,3.1,1,0.684012,0.497115,0.266474,South
1,2,Male,64,2,Bi-Weekly,26,192,Bronze,3.1,1,0.684012,0.376474,0.143590,Northeast
2,3,Male,73,23,Weekly,52,1752,Gold,3.1,1,0.684012,0.691026,0.436667,Northeast
3,4,Male,90,49,Weekly,52,4500,Platinum,3.5,0,0.684012,0.951987,0.692500,Northeast
4,5,Male,49,31,Annually,1,1568,Gold,2.7,1,0.684012,0.344423,0.328077,West
5,6,Male,20,14,Weekly,52,300,Bronze,2.9,1,0.684012,0.601474,0.125577,West
6,7,Male,85,49,Quarterly,4,4250,Platinum,3.2,1,0.684012,0.633397,0.625577,West
7,8,Male,34,19,Weekly,52,680,Silver,3.2,1,0.684012,0.650064,0.266538,South
8,9,Male,97,8,Annually,1,873,Silver,2.6,1,0.684012,0.114936,0.182628,South
9,10,Male,31,4,Quarterly,4,155,Bronze,4.8,0,0.684012,0.184359,0.478397,Midwest


In [13]:
# PART 4: SANITY CHECKS ON ENGINEERED FEATURES

print("\n" + "="*80)
print("4.1 FEATURE SANITY CHECKS")
print("="*80)
print("\nValue Tier counts (should be ~roughly even due to qcut):")
print(df_clean['Value_Tier'].value_counts())

print("\nPrevious Purchases at the boundary (check for censoring at max=50):")
print("Customers with Previous Purchases == 50:", (df_clean['Previous Purchases'] == 50).sum())
print(" -> If this is a meaningfully large group, your top tenure bucket may be")
print("    censored (i.e., real tenure could be higher than the data shows).")

print("\nLoyalty A vs Loyalty B — correlation (expect low/moderate, not 1.0):")
print(df_clean[['Loyalty_Score_A_Behavioral', 'Loyalty_Score_B_Value_Satisfaction']].corr())

print("\nCross-tab: customers high (top 25%) on both definitions vs. only one")
high_a = df_clean['Loyalty_Score_A_Behavioral'] >= 0.75
high_b = df_clean['Loyalty_Score_B_Value_Satisfaction'] >= 0.75
print("High on both A and B (safest loyal segment):", (high_a & high_b).sum())
print("High on A only (behaviorally loyal, but value/satisfaction lower):", (high_a & ~high_b).sum())
print("High on B only (valuable/satisfied, but not yet habitual — growth target):", (~high_a & high_b).sum())
print(" -> This disagreement zone is the actual insight: it identifies customers")
print("    who look loyal on one lens but not the other, which a single score would hide.")

print("\nPromo Dependency Score by Value Tier (does reliance differ by value?):")
print(df_clean.groupby('Value_Tier')['Promo_Dependency_Score'].mean())

# Final cleaned + engineered dataset, ready for SQL / Power BI layers
print("\n" + "="*80)
print("FINAL COLUMNS")
print("="*80)
print(df_clean.columns.tolist())

df_clean.to_csv("customer_features_engineered.csv", index=False)


4.1 FEATURE SANITY CHECKS

Value Tier counts (should be ~roughly even due to qcut):
Value_Tier
Silver      980
Bronze      975
Platinum    974
Gold        971
Name: count, dtype: int64

Previous Purchases at the boundary (check for censoring at max=50):
Customers with Previous Purchases == 50: 77
 -> If this is a meaningfully large group, your top tenure bucket may be
    censored (i.e., real tenure could be higher than the data shows).

Loyalty A vs Loyalty B — correlation (expect low/moderate, not 1.0):
                                    Loyalty_Score_A_Behavioral  Loyalty_Score_B_Value_Satisfaction
Loyalty_Score_A_Behavioral                            1.000000                            0.408452
Loyalty_Score_B_Value_Satisfaction                    0.408452                            1.000000

Cross-tab: customers high (top 25%) on both definitions vs. only one
High on both A and B (safest loyal segment): 144
High on A only (behaviorally loyal, but value/satisfaction lower): 333
H

/tmp/ipykernel_58/3806744388.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df_clean.groupby('Value_Tier')['Promo_Dependency_Score'].mean())


In [16]:
# A customer who's behaviorally loyal but lower value/satisfaction
df_clean[(df_clean['Loyalty_Score_A_Behavioral'] >= 0.75) & 
          (df_clean['Loyalty_Score_B_Value_Satisfaction'] < 0.5)][preview_cols].head(5)

# A customer who's high value/satisfaction but not yet habitual
df_clean[(df_clean['Loyalty_Score_B_Value_Satisfaction'] >= 0.75) & 
          (df_clean['Loyalty_Score_A_Behavioral'] < 0.5)][preview_cols].head(5)

,Customer ID,Gender,Purchase Amount (USD),Previous Purchases,Frequency of Purchases,Purchases_Per_Year,Estimated_Lifetime_Spend,Value_Tier,Review Rating,Low_Satisfaction_Flag,Promo_Dependency_Score,Loyalty_Score_A_Behavioral,Loyalty_Score_B_Value_Satisfaction,Region
32,33,Male,67,37,Annually,1,2546,Platinum,4.9,0,0.684012,0.403846,0.881026,Midwest
40,41,Male,76,31,Quarterly,4,2432,Platinum,4.6,0,0.684012,0.454615,0.811795,South
91,92,Male,99,18,Every 3 Months,4,1881,Gold,4.6,0,0.684012,0.321090,0.757436,South
97,98,Male,92,37,Annually,1,3496,Platinum,4.8,0,0.684012,0.403846,0.922885,South
102,103,Male,67,35,Quarterly,4,2412,Platinum,4.8,0,0.684012,0.495897,0.849615,South


In [5]:
# ============================================================================
# DECODING CUSTOMER VALUE — Step 1: EDA, Cleaning & Feature Engineering
# ============================================================================
# Update DATA_PATH to wherever the CSV sits in your Kaggle notebook.

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

DATA_PATH = "/kaggle/input/datasets/grachna/decoding-customer-value-strategy-project/Dataset.csv"   # <-- update path in Kaggle
df = pd.read_csv(DATA_PATH)

# ============================================================================
# PART 1: EXPLORATORY DATA ANALYSIS
# Goal: understand structure, find anomalies, and surface the structural
# artifacts that will shape every decision in Part 2 and 3.
# ============================================================================

print("="*80)
print("1.1 BASIC STRUCTURE")
print("="*80)
print("Shape:", df.shape)
print("Customer ID unique:", df['Customer ID'].is_unique)
print("\nDtypes:\n", df.dtypes)
print("\nNulls:\n", df.isnull().sum())
print("\nFull duplicate rows:", df.duplicated().sum())

print("\n" + "="*80)
print("1.2 NUMERIC SUMMARY")
print("="*80)
print(df[['Age', 'Purchase Amount (USD)', 'Review Rating', 'Previous Purchases']].describe())

print("\n" + "="*80)
print("1.3 CATEGORICAL DISTRIBUTIONS")
print("="*80)
cat_cols = ['Gender', 'Category', 'Season', 'Subscription Status', 'Shipping Type',
            'Discount Applied', 'Promo Code Used', 'Payment Method', 'Frequency of Purchases']
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())

print("\n" + "="*80)
print("1.4 STRUCTURAL ARTIFACT CHECKS (critical — these drive cleaning decisions)")
print("="*80)

# Check 1: Are Discount Applied and Promo Code Used actually two different fields?
match_rate = (df['Discount Applied'] == df['Promo Code Used']).mean()
print(f"Discount Applied == Promo Code Used match rate: {match_rate:.2%}")
print(" -> If ~100%, these are duplicate fields. Keep one, drop the other, document why.")

# Check 2: Is Discount Applied actually a behavioral signal, or a structural flag?
print("\nDiscount Applied rate by Gender:")
print(df.groupby('Gender')['Discount Applied'].apply(lambda x: (x == 'Yes').mean()))

print("\nDiscount Applied rate by Subscription Status:")
print(df.groupby('Subscription Status')['Discount Applied'].apply(lambda x: (x == 'Yes').mean()))

print("\nSubscription Status by Gender (check if subscription itself is gender-confounded):")
print(pd.crosstab(df['Gender'], df['Subscription Status'], normalize='index'))
print(" -> If discount usage is ~0%/100% split by gender or subscription, it is NOT a")
print("    customer choice signal in this dataset — it's a generation artifact.")
print("    Decision: exclude raw Discount Applied from value/loyalty scoring, or only")
print("    use it stratified within gender. State this explicitly as a data limitation.")

# Check 3: Are Bi-Weekly and Fortnightly being treated as different cadences
# when they mean the same thing (every two weeks)?
print("\nFrequency of Purchases categories (check for duplicate-meaning labels):")
print(df['Frequency of Purchases'].value_counts())
print(" -> 'Bi-Weekly' and 'Fortnightly' both mean every 2 weeks. Must be merged")
print("    before converting frequency to a numeric scale, or you'll artificially")
print("    split your highest-cadence segment in two.")

print("\n" + "="*80)
print("1.5 CORRELATION CHECK (sets expectations for feature engineering)")
print("="*80)
print(df[['Age', 'Purchase Amount (USD)', 'Review Rating', 'Previous Purchases']].corr())
print(" -> If these are all near-zero, no single raw variable will tell the loyalty")
print("    story on its own. Composite/engineered features are necessary, not optional.")

print("\n" + "="*80)
print("1.6 MAIN-EFFECT CHECKS: Category / Season / Geography vs tenure & spend")
print("="*80)
print("\nAvg Previous Purchases by Category:")
print(df.groupby('Category')['Previous Purchases'].mean().sort_values(ascending=False))
print("\nAvg Previous Purchases by Season:")
print(df.groupby('Season')['Previous Purchases'].mean().sort_values(ascending=False))
print("\nAvg spend by Location (top 8 / bottom 8) — flag if sample sizes are small:")
loc_summary = df.groupby('Location').agg(
    n=('Customer ID', 'count'),
    avg_spend=('Purchase Amount (USD)', 'mean'),
    avg_prev_purchases=('Previous Purchases', 'mean')
).sort_values('avg_spend', ascending=False)
print(loc_summary.head(8))
print(loc_summary.tail(8))
print(" -> ~50 states with ~60-90 customers each means state-level claims are noisy.")
print("    Roll up to region for the 'geographic opportunity' analysis; use state only")
print("    for illustrative drill-down, with sample size caveated.")

print("\n" + "="*80)
print("1.7 MISSINGNESS PATTERN — Review Rating")
print("="*80)
print("Missing count:", df['Review Rating'].isnull().sum())
print("Missingness by Category (check if missing is random or structural):")
print(df[df['Review Rating'].isnull()]['Category'].value_counts(normalize=True))
print(df['Category'].value_counts(normalize=True))
print(" -> Compare the two distributions above. If similar, missingness looks random")
print("    -> safe to impute. If skewed, document as a limitation.")


# ============================================================================
# PART 2: CLEANING
# Every step here is a documented decision, not a silent transformation.
# ============================================================================

df_clean = df.copy()

# Decision 1: Discount Applied and Promo Code Used are identical -> drop one.
# Keep 'Discount Applied' as canonical; drop 'Promo Code Used' to avoid double-
# counting the same signal in any downstream score.
assert (df_clean['Discount Applied'] == df_clean['Promo Code Used']).mean() > 0.99, \
    "Re-check: these were expected to be near-identical."
df_clean = df_clean.drop(columns=['Promo Code Used'])

# Decision 2: Merge Bi-Weekly and Fortnightly (same real-world cadence) before
# building any numeric frequency feature.
df_clean['Frequency of Purchases'] = df_clean['Frequency of Purchases'].replace(
    {'Fortnightly': 'Bi-Weekly'}
)

# Decision 3: Review Rating — add a missingness flag (whether a customer bothered
# to rate at all can itself be a weak engagement signal), then impute with the
# category median so we don't lose 37 rows from analysis.
df_clean['Rating_Missing_Flag'] = df_clean['Review Rating'].isnull().astype(int)
df_clean['Review Rating'] = df_clean.groupby('Category')['Review Rating'] \
    .transform(lambda x: x.fillna(x.median()))

# Decision 4: Standardize Yes/No fields to boolean for cleaner downstream math.
for col in ['Discount Applied', 'Subscription Status']:
    df_clean[col + '_Flag'] = (df_clean[col] == 'Yes').astype(int)

print("\n" + "="*80)
print("2.1 CLEANING SUMMARY")
print("="*80)
print("Shape after cleaning:", df_clean.shape)
print("Remaining nulls:\n", df_clean.isnull().sum().sum())


# ============================================================================
# PART 3: FEATURE ENGINEERING
# Every feature below is tied to a specific business question — not built
# because it was easy to compute. The "why" is in the comment above each one.
# ============================================================================

# ---- 3.1 Estimated Lifetime Spend ----
# WHY: There's no true lifetime spend field — only a single observed order
# amount + a count of previous purchases. We approximate lifetime spend by
# assuming the observed order value is representative of the customer's
# typical order. This is a load-bearing assumption: state it explicitly in
# the report. It answers: "how much revenue has this customer likely
# generated for us so far?"
df_clean['Estimated_Lifetime_Spend'] = (
    df_clean['Purchase Amount (USD)'] * (df_clean['Previous Purchases'] + 1)
)

# ---- 3.2 Purchase Frequency (numeric, purchases/year) ----
# WHY: 'Frequency of Purchases' is an ordinal label, not a number, which makes
# it unusable in any score or ranking. Converting to an estimated annual
# purchase rate lets us combine it with spend and tenure. Answers: "how
# active is this customer, on a comparable numeric scale?"
freq_map = {
    'Weekly': 52,
    'Bi-Weekly': 26,
    'Monthly': 12,
    'Quarterly': 4,
    'Every 3 Months': 4,
    'Annually': 1
}
df_clean['Purchases_Per_Year'] = df_clean['Frequency of Purchases'].map(freq_map)

# ---- 3.3 Value Tier ----
# WHY: A continuous value score is hard for a non-technical founder to act on.
# Tiering (quartile-based) directly answers: "which customers should get
# white-glove treatment vs. which are low priority?" — a decision, not just
# a number. Built from Estimated Lifetime Spend, since that's the closest
# proxy we have to "how much has this person been worth to us."
df_clean['Value_Tier'] = pd.qcut(
    df_clean['Estimated_Lifetime_Spend'],
    q=4,
    labels=['Bronze', 'Silver', 'Gold', 'Platinum']
)

# ---- 3.4 Satisfaction Flag ----
# WHY: Answers a sharp, specific question the brand needs flagged: "do we
# have high-value customers who are quietly unhappy?" Rather than just
# reporting average rating, this creates an actionable binary the marketing
# team can filter on directly.
df_clean['Low_Satisfaction_Flag'] = (df_clean['Review Rating'] < 3.5).astype(int)

# ---- 3.5 Promo Dependency Score (handled carefully due to the gender/
# subscription artifact found in EDA) ----
# WHY: The brand explicitly wants to know who needs a discount to buy. But we
# found Discount Applied is structurally tied to Gender (0% for Female, ~63%
# for Male) and Subscription (100% when subscribed) — not a clean behavioral
# signal. Building this as a raw flag would let a data artifact masquerade as
# a business insight. Instead, we rank discount usage WITHIN gender, so the
# score reflects relative reliance among comparable customers, not the
# underlying structural skew. This must be stated as a limitation in the
# report regardless.
df_clean['Promo_Dependency_Score'] = df_clean.groupby('Gender')['Discount Applied_Flag'] \
    .rank(pct=True)

# ---- 3.6 Loyalty Definition A — Behavioral Loyalty (Engagement Depth) ----
# WHY: Originally this blended Previous Purchases with Purchases_Per_Year.
# Validation check: correlation between Frequency-derived cadence and actual
# Previous Purchases is ~0.01 (essentially zero), and average Previous
# Purchases is flat (~24-27) across every single frequency label, including
# 36% of "Annually" shoppers having 30+ previous purchases. Subscription
# Status tested the same way (even within Male only, to remove the gender
# confound) also shows ~0 correlation with Previous Purchases (r=0.02).
# Conclusion: neither field carries real signal about purchase history in
# this dataset. Blending them into the score would dilute a clean signal
# with noise — exactly the "sounds analytical but isn't" trap the brief
# warns against. Previous Purchases is the ONLY field in this dataset
# empirically validated as a genuine, uncontaminated behavioral signal, so
# Loyalty A is built on it alone. This mirrors the "Frequency" component of
# classic RFM analysis and stays a genuinely distinct lens from Loyalty B
# (which is value + satisfaction based) rather than two scores secretly
# built from overlapping noise.
# Address the censoring anomaly at the boundary (Previous Purchases caps at
# 50; ~2% of customers sit exactly at this ceiling, meaning their true tenure
# is likely understated but tied with each other). We do NOT use Frequency
# of Purchases to resolve this, since it has no proven relationship to
# tenure (r=0.01) and would inject noise rather than correct anything.
# Instead we break ties only WITHIN the capped group using
# Estimated_Lifetime_Spend as a secondary sort key — this only reorders the
# ~77 tied customers relative to each other; it does not change anyone
# else's rank or blend an unvalidated variable into the main score.
df_clean['Previous_Purchases_TieBreak'] = (
    df_clean['Previous Purchases'].astype(float) +
    (df_clean['Estimated_Lifetime_Spend'].rank(pct=True) * 0.0001)
    # tiny fractional nudge: only changes ordering among exact ties,
    # cannot move a customer past anyone with a genuinely different
    # Previous Purchases value
)
df_clean['Tenure_Censored_Flag'] = (df_clean['Previous Purchases'] == 50).astype(int)

df_clean['Loyalty_Score_A_Behavioral'] = df_clean['Previous_Purchases_TieBreak'].rank(pct=True)

# Tiered version for business-facing use (raw percentiles aren't actionable
# on their own — a marketing team needs labeled buckets to target against).
df_clean['Engagement_Depth_Tier'] = pd.qcut(
    df_clean['Loyalty_Score_A_Behavioral'],
    q=4,
    labels=['New', 'Developing', 'Established', 'Veteran']
)

# Note: Purchases_Per_Year is kept as a standalone feature (useful for the
# SQL segmentation question on seasonal/category patterns by stated
# cadence), but it is NOT used in any loyalty/value score, since it failed
# the correlation validation above.

# ---- 3.6b Frequency/Tenure Inconsistency Flag (diagnostic only) ----
# WHY: EDA surfaced cases like a customer reporting "Annually" purchasing
# while having 37 Previous Purchases — implausible if taken literally. We
# tested whether Frequency could be used to adjust/interpolate Previous
# Purchases and confirmed it cannot: a multivariate regression controlling
# for Age, Gender, and Category showed Frequency adds no significant
# explanatory power for Previous Purchases (p=0.118, R² contribution
# ~0.002). Using a variable with no validated relationship to "correct"
# another would manufacture false consistency, not real consistency.
# Instead, we flag the inconsistency explicitly as a documented data
# quality finding -- useful for the report's limitations section -- without
# altering the trusted Previous Purchases column or any score built on it.
implausible = (
    ((df_clean['Frequency of Purchases'] == 'Annually') & (df_clean['Previous Purchases'] > 20)) |
    ((df_clean['Frequency of Purchases'] == 'Weekly') & (df_clean['Previous Purchases'] < 5))
)
df_clean['Frequency_Inconsistency_Flag'] = implausible.astype(int)
print(f"\n{implausible.sum()} customers ({implausible.mean():.1%}) show a frequency label "
      f"implausible given their purchase count -- flagged as a data quality limitation, "
      f"not used to alter Previous Purchases or any score.")

# ---- 3.7 Loyalty Definition B — Value + Satisfaction Loyalty ----
# WHY: Tests a different lens — are big spenders also satisfied? Answers:
# "is this customer's value sustainable, or are they a high-spender who
# might churn because they're unhappy?" This surfaces a segment Definition A
# alone would miss: high spend, low satisfaction = flight risk.
df_clean['Loyalty_Score_B_Value_Satisfaction'] = (
    df_clean['Estimated_Lifetime_Spend'].rank(pct=True) * 0.5 +
    df_clean['Review Rating'].rank(pct=True) * 0.5
)

def _assign_segment(row):
    high_a = row['Loyalty_Score_A_Behavioral'] >= 0.75
    high_b = row['Loyalty_Score_B_Value_Satisfaction'] >= 0.75
    if high_a and high_b:
        return 'Core Loyal'
    if high_a and not high_b:
        return 'Habitual, Not Yet Valuable'
    if not high_a and high_b:
        return 'Promising New'
    return 'Low Priority / At Risk'

df_clean['Customer_Segment'] = df_clean.apply(_assign_segment, axis=1)
# ---- 3.8 Region rollup (for geography analysis) ----
# WHY: ~50 states with only 60-90 customers each is too noisy for state-level
# claims. Rolling up to census region gives a more statistically stable view
# for the "underlevered geography" question; state-level can still be used
# for illustrative drill-downs with sample-size caveats.
region_map = {
    'Connecticut': 'Northeast', 'Maine': 'Northeast', 'Massachusetts': 'Northeast',
    'New Hampshire': 'Northeast', 'Rhode Island': 'Northeast', 'Vermont': 'Northeast',
    'New Jersey': 'Northeast', 'New York': 'Northeast', 'Pennsylvania': 'Northeast',
    'Illinois': 'Midwest', 'Indiana': 'Midwest', 'Michigan': 'Midwest', 'Ohio': 'Midwest',
    'Wisconsin': 'Midwest', 'Iowa': 'Midwest', 'Kansas': 'Midwest', 'Minnesota': 'Midwest',
    'Missouri': 'Midwest', 'Nebraska': 'Midwest', 'North Dakota': 'Midwest', 'South Dakota': 'Midwest',
    'Delaware': 'South', 'Florida': 'South', 'Georgia': 'South', 'Maryland': 'South',
    'North Carolina': 'South', 'South Carolina': 'South', 'Virginia': 'South',
    'West Virginia': 'South', 'Alabama': 'South', 'Kentucky': 'South', 'Mississippi': 'South',
    'Tennessee': 'South', 'Arkansas': 'South', 'Louisiana': 'South', 'Oklahoma': 'South',
    'Texas': 'South',
    'Arizona': 'West', 'Colorado': 'West', 'Idaho': 'West', 'Montana': 'West',
    'Nevada': 'West', 'New Mexico': 'West', 'Utah': 'West', 'Wyoming': 'West',
    'Alaska': 'West', 'California': 'West', 'Hawaii': 'West', 'Oregon': 'West',
    'Washington': 'West'
}
df_clean['Region'] = df_clean['Location'].map(region_map)
print("\nUnmapped locations (check if any state names are missing from region_map):")
print(df_clean[df_clean['Region'].isnull()]['Location'].unique())

# ============================================================================
# PART 4: SANITY CHECKS ON ENGINEERED FEATURES
# Never trust an engineered feature without checking its distribution.
# ============================================================================

print("\n" + "="*80)
print("4.1 FEATURE SANITY CHECKS")
print("="*80)
print("\nValue Tier counts (should be ~roughly even due to qcut):")
print(df_clean['Value_Tier'].value_counts())

print("\nPrevious Purchases at the boundary (check for censoring at max=50):")
print("Customers with Previous Purchases == 50:", (df_clean['Previous Purchases'] == 50).sum())
print(" -> If this is a meaningfully large group, your top tenure bucket may be")
print("    censored (i.e., real tenure could be higher than the data shows).")

print("\nLoyalty A vs Loyalty B — correlation (expect low/moderate, not 1.0):")
print(df_clean[['Loyalty_Score_A_Behavioral', 'Loyalty_Score_B_Value_Satisfaction']].corr())

print("\nCross-tab: customers high (top 25%) on both definitions vs. only one")
high_a = df_clean['Loyalty_Score_A_Behavioral'] >= 0.75
high_b = df_clean['Loyalty_Score_B_Value_Satisfaction'] >= 0.75
print("High on both A and B (safest loyal segment):", (high_a & high_b).sum())
print("High on A only (behaviorally loyal, but value/satisfaction lower):", (high_a & ~high_b).sum())
print("High on B only (valuable/satisfied, but not yet habitual — growth target):", (~high_a & high_b).sum())
print(" -> This disagreement zone is the actual insight: it identifies customers")
print("    who look loyal on one lens but not the other, which a single score would hide.")

print("\nPromo Dependency Score by Value Tier (does reliance differ by value?):")
print(df_clean.groupby('Value_Tier')['Promo_Dependency_Score'].mean())

# ============================================================================
# Final cleaned + engineered dataset, ready for SQL / Power BI layers
# ============================================================================
print("\n" + "="*80)
print("FINAL COLUMNS")
print("="*80)
print(df_clean.columns.tolist())
df_clean.to_csv("customer_features_engineered.csv", index=False)

def assign_segment(row):
    high_a = row['Loyalty_Score_A_Behavioral'] >= 0.75
    high_b = row['Loyalty_Score_B_Value_Satisfaction'] >= 0.75
    if high_a and high_b: return 'Core Loyal'
    if high_a and not high_b: return 'Habitual, Not Yet Valuable'
    if not high_a and high_b: return 'Promising New'
    return 'Low Priority / At Risk'

df_clean['Customer_Segment'] = df_clean.apply(assign_segment, axis=1)


1.1 BASIC STRUCTURE
Shape: (3900, 18)
Customer ID unique: True

Dtypes:
 Customer ID                 int64
Age                         int64
Gender                     object
Item Purchased             object
Category                   object
Purchase Amount (USD)       int64
Location                   object
Size                       object
Color                      object
Season                     object
Review Rating             float64
Subscription Status        object
Shipping Type              object
Discount Applied           object
Promo Code Used            object
Previous Purchases          int64
Payment Method             object
Frequency of Purchases     object
dtype: object

Nulls:
 Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Re

/tmp/ipykernel_58/1507165864.py:362: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df_clean.groupby('Value_Tier')['Promo_Dependency_Score'].mean())
